# Mission 06 — Part 3: Data & Payload Specialist
## MDP Trajectory Logger, Empirical Return & Infinite Loop Sentinel (USSE Noosphere, `SEC-06-CAR`)

Implementation: `03_MDP_Trajectory_Logger.py` and `03_Infinite_Loop_Sentinel.py` (standard library only). This section covers: trajectory logging $\tau=(s_0,a_0,r_0,\dots,s_T)$, the discounted return $G_0=\sum_t\gamma^t r_t$ with $\gamma=0.95$, the empirical transition matrix $P(s'\mid s,a)$, and the SHA-256 state-history sentinel with Floyd cycle detection.

> **Data note.** The briefing mentions a *provided* dataset of past jumps; it is not in the course repository. The jump log used below is a **mock**, written by hand so each probability can be verified by counting. The loader `load_jumps_csv` accepts the real file (`state,action,next_state`) when it exists.

In [1]:
import importlib.util, math, pathlib

def load(name):
    spec = importlib.util.spec_from_file_location(name, pathlib.Path(f"{name}.py"))
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

mdp = load("03_MDP_Trajectory_Logger")
sentinel = load("03_Infinite_Loop_Sentinel")
print("loaded:", mdp.TrajectoryLogger.__name__, "|", sentinel.InfiniteLoopSentinel.__name__)

loaded: TrajectoryLogger | InfiniteLoopSentinel


### 1. Trajectory logging
`TrajectoryLogger.start_episode(s0)` opens an episode; every executed action is recorded with `log_step(action, reward, next_state)`; `end_episode()` freezes it into a `Trajectory`. The deliberation loop (Part 4) calls these three methods.

In [2]:
log = mdp.TrajectoryLogger(gamma=0.95)
log.start_episode("s0")
for action, reward, nxt in [("a0", 10, "s1"), ("a1", 0, "s2"), ("a2", -5, "s3"), ("a3", 20, "s4")]:
    log.log_step(action, reward, nxt)
tau = log.end_episode()
print("tau =", tau.as_sequence())
print("T =", tau.T, "steps,", len(tau.states), "states")

tau = ['s0', 'a0', 10.0, 's1', 'a1', 0.0, 's2', 'a2', -5.0, 's3', 'a3', 20.0, 's4']
T = 4 steps, 5 states


### 2. Empirical discounted return $G_0$ — master test 3 (MDP Return Test)
$G_0=\sum_{t=0}^{T}\gamma^t r_t$ is evaluated backwards, $G_t=r_t+\gamma G_{t+1}$. For rewards `[10, 0, -5, 20]` and $\gamma=0.95$: $10+0.95^2\cdot(-5)+0.95^3\cdot 20=22.635$.

In [3]:
rewards, gamma = [10, 0, -5, 20], 0.95
expected = 10 + gamma**0 * 0 + gamma**2 * -5 + gamma**3 * 20      # term by term, independent of the implementation
g0 = mdp.discounted_return(rewards, gamma)
assert math.isclose(g0, expected, abs_tol=1e-12) and math.isclose(g0, 22.635, abs_tol=1e-9)
print(f"G_0 = {g0:.6f}   (expected {expected:.6f})   MDP Return Test PASSED")
print("G_t for every step:", [round(g, 4) for g in mdp.returns_to_go(rewards, gamma)])
print()
mdp.demo_episodes();

G_0 = 22.635000   (expected 22.635000)   MDP Return Test PASSED
G_t for every step: [22.635, 13.3, 14.0, 20.0]

Demo episodes (illustrative rewards)
  System 1 reflex    : T=2  rewards=[5.0, -95.0]  G_0(gamma=0.95)= -85.250  G_0(gamma=0, first reward only)=  5.0
  System 2 deliberate: T=3  rewards=[-1.0, -1.0, 99.0]  G_0(gamma=0.95)=  87.397  G_0(gamma=0, first reward only)= -1.0
  A short-sighted agent (gamma=0) prefers the reflex; at gamma=0.95 the deliberate plan wins.


### 3. Transition frequency matrix $\hat P(s'\mid s,a)=N(s,a,s')/N(s,a)$
Rows are the observed `(state, action)` pairs, columns the observed next states, and each row sums to 1. A pair that never occurs in the log has **no** estimate (`KeyError`), which is not the same as probability 0.

In [4]:
model = mdp.TransitionModel.from_jumps(mdp.JUMP_LOG)
print(f"{model.total_jumps} jumps (MOCK log)")
print(model.format_table())
rows, cols, m = model.matrix()
assert all(math.isclose(sum(r), 1.0, abs_tol=1e-12) for r in m)
print()
print("P(escape_corridor | dust_lane, warp_high_shields) =", model.probability_of("dust_lane", "warp_high_shields", ["escape_corridor"]))
print("P(hull_breach     | plasma_shear, warp_low_shields) =", model.prob("plasma_shear", "warp_low_shields", "hull_breach"))

56 jumps (MOCK log)
                                           dust_lane escape_corridor homunculus_core     hull_breach  outer_filament    plasma_shear      N
dust_lane | warp_high_shields                  0.000           0.600           0.300           0.100           0.000           0.000     10
dust_lane | warp_low_shields                   0.000           0.000           0.500           0.500           0.000           0.000      4
homunculus_core | hold_and_scan                0.000           0.000           0.667           0.333           0.000           0.000      3
homunculus_core | warp_high_shields            0.000           0.571           0.000           0.429           0.000           0.000      7
outer_filament | warp_high_shields             0.300           0.000           0.000           0.000           0.100           0.600     10
outer_filament | warp_low_shields              0.000           0.000           0.000           0.400           0.000           0.600      5


### 4. Infinite Loop Sentinel — master test 4 (Sentinel Cycle Test)
Each step is hashed (SHA-256 of state + action + arguments). The tripwire fires after three identical steps in a row, or when Floyd's tortoise and hare finds a cycle and the current state+action has been seen three times. The starter's consecutive counter alone would never see the loop $s_1\to s_2\to s_3\to s_1$.

In [5]:
seq = [("s1", "a1"), ("s2", "a2"), ("s3", "a3")] * 4
starter_style = sentinel.InfiniteLoopSentinel()
print("consecutive-only check_loop() flags the cycle over", len(seq), "steps:", any(starter_style.check_loop(a, {}, state=s)[0] for s, a in seq))

guard, halted_at = sentinel.InfiniteLoopSentinel(repeat_threshold=3), None
for step, (state, action) in enumerate(seq, start=1):
    report = guard.observe(action, {}, state=state)
    if report.tripped:
        halted_at = step
        break
assert halted_at is not None and halted_at < 10
assert report.cycle == (0, 3)
print(f"Floyd cycle (start index {report.cycle[0]}, length {report.cycle[1]}); tripwire at step {halted_at}  ->  Sentinel Cycle Test PASSED")

SECURITY VIOLATION: infinite loop detected at step 7 (cycle of length 3 starting at step 1; this state+action seen 3x): action=a1 state=s1


consecutive-only check_loop() flags the cycle over 12 steps: False
Floyd cycle (start index 0, length 3); tripwire at step 7  ->  Sentinel Cycle Test PASSED


### 5. Full self-test suites
Both files run the same checks as a script (`python 03_MDP_Trajectory_Logger.py`, `python 03_Infinite_Loop_Sentinel.py`). Test 7 of the sentinel suite runs the sentinel against Part 1's `Agent` when `01_Agent_State_Machine*.py` is next to it.

In [6]:
mdp.run_self_tests()
print()
sentinel.run_self_tests()

[TEST 1] MDP Return Test (briefing): rewards [10, 0, -5, 20], gamma = 0.95
         G_0 = 22.635000  (expected 22.635000)  PASS
[TEST 2] discounted_return edge cases and returns_to_go
         PASS
[TEST 3] TrajectoryLogger records tau = (s0, a0, r0, ..., sT)
         PASS
[TEST 4] TransitionModel on the mock jump log (hand-counted values)
         PASS
[SUCCESS] Part 3 (MDP logger, G_0, transition matrix) self-tests passed.

[TEST 1] SHA-256 step hash is deterministic and covers state + action + arguments
         PASS
[TEST 2] starter interface: check_loop() counts consecutive repeats
         PASS
[TEST 3] Floyd's tortoise and hare agrees with brute force (300 random trajectories)
         PASS
[TEST 4] Sentinel Cycle Test (briefing): [s1, s2, s3, s1, s2, s3, ...] trips before step 10
         tripwire fired at step 7: cycle of length 3 starting at step 1; this state+action seen 3x   PASS
[TEST 5] no false alarms on a trajectory that never repeats or only varies
         PASS
[TEST 

### Notes
* **Mock data:** the jump log and the two demo episodes are illustrative; no claim about the real nebula dataset is made. The empirical probabilities are maximum-likelihood estimates from 56 jumps, so rows with few observations (for example `homunculus_core | hold_and_scan`, N = 3) are noisy.
* **Horizon:** $T$ steps give $T$ rewards ($r_0,\dots,r_{T-1}$) and $T+1$ states. `discounted_return` sums every reward it is given, which is what the master test expects.
* **Tripwire:** the Floyd branch assumes a deterministic loop (the same state+action is always followed by the same next one); otherwise only the three-in-a-row rule applies. Hash the state as well as the action to avoid mistaking different situations for a loop.
* **Overlap:** the briefing assigns cycle detection to the Security & Ethics role, while the mission README and starter file `03_Infinite_Loop_Sentinel_starter.py` assign the sentinel to this role. Both are covered here; keep whichever the squad agrees on in the final notebook.